# Phi-3 Colab Output Generation Only

Notebook này chạy trên Google Colab, mount Google Drive để lấy model/cache, clone GitHub repo để lấy scripts và dataset, rồi sinh raw outputs cho 2 model Phi-3 theo format giống `outputs/llama4/*_outputs.json`. Notebook này không chấm điểm và không gọi LLM judge.

## 1. Runtime

Trên Colab, chọn `Runtime > Change runtime type > GPU`. Với Phi-3 Medium, nên dùng GPU RAM cao nếu có. T4 vẫn có thể thử với 4-bit quantization, nhưng dễ thiếu VRAM nếu session đã giữ nhiều bộ nhớ.

In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/a-triple-of-lms") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive")

if IN_COLAB:
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if not (REPO_DIR / "src" / "phi3.py").exists():
    raise FileNotFoundError(f"Repo/script not found: {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print("IN_COLAB:", IN_COLAB)
print("Repo:", REPO_DIR)
print("Drive:", DRIVE_ROOT if IN_COLAB else "not mounted")

In [ ]:
# Install runtime dependencies. Safe to rerun.
packages = [
    "transformers>=4.41.0",
    "accelerate>=0.30.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "einops",
    "tqdm",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)
print("Dependencies installed")

## 2. Cấu hình model/cache

Sửa `HF_CACHE_DIR` nếu model của bạn nằm ở thư mục khác trên Drive. Nếu bạn lưu model thành thư mục riêng, đặt `local_path` tương ứng trong `MODEL_SPECS`. Nếu dùng Hugging Face cache chuẩn trên Drive, để `local_path = None` và `model_id` là repo id.

In [ ]:
DATA_FILE = REPO_DIR / "data" / "bigdata_10_questions.json"
OUTPUT_DIR = REPO_DIR / "outputs" / "phi3"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "a-triple-of-lms" / "outputs" / "phi3"

# Folder chứa HF cache trên Drive. Có thể là folder chứa `hub/` hoặc chứa trực tiếp `models--*`.
HF_CACHE_DIR = DRIVE_ROOT / "hf_cache"

LOCAL_FILES_ONLY = True       # True = chỉ lấy model từ Drive/cache, không download thêm
LOAD_IN_4BIT = True           # Recommended for Colab GPU
DEVICE = "auto"
TORCH_DTYPE = "float16"
LIMIT = 0                     # 0 = full benchmark; đặt 1-2 để smoke test
SLEEP_SECONDS = 0.0
COPY_OUTPUTS_TO_DRIVE = True

MODEL_SPECS = [
    {
        "key": "mini",
        "model_name": "Phi-3 Mini 4K",
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "local_path": None,
        "trust_remote_code": False,
    },
    {
        "key": "medium",
        "model_name": "Phi-3 Medium 4K",
        "model_id": "microsoft/Phi-3-medium-4k-instruct",
        "local_path": None,
        "trust_remote_code": False,
    },
]

print("Data:", DATA_FILE)
print("Output:", OUTPUT_DIR)
print("HF cache:", HF_CACHE_DIR)

In [ ]:
from tqdm import tqdm

from src.phi3 import (
    SYSTEM_PROMPT,
    build_chat_prompt,
    infer_input_device,
    load_phi3_model,
    load_samples,
    normalize_hub_cache_dir,
    parse_generation_params,
    safe_filename,
)

cache_dir = normalize_hub_cache_dir(str(HF_CACHE_DIR)) if HF_CACHE_DIR else None
samples = load_samples(DATA_FILE)
if LIMIT:
    samples = samples[:LIMIT]

print("Samples:", len(samples))
print("Resolved cache_dir:", cache_dir)

In [ ]:
def model_ref(spec: dict) -> str:
    local_path = spec.get("local_path")
    if local_path:
        path = Path(local_path)
        if path.exists():
            return str(path)
        raise FileNotFoundError(f"local_path not found for {spec['model_name']}: {path}")
    return spec["model_id"]


def save_json(data, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
        file.write("\n")
    print("Saved:", path)


def generate_phi3_response(tokenizer, model, prompt: str, temperature: float, max_tokens: int):
    import torch

    text = build_chat_prompt(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt")
    device = infer_input_device(model)
    inputs = {key: value.to(device) for key, value in inputs.items()}
    prompt_tokens = int(inputs["input_ids"].shape[-1])

    generate_kwargs = {
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generate_kwargs.update({"do_sample": True, "temperature": temperature})
    else:
        generate_kwargs["do_sample"] = False

    started_at = datetime.now(timezone.utc).isoformat()
    start_timer = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **generate_kwargs)
    latency_s = round(time.perf_counter() - start_timer, 3)
    ended_at = datetime.now(timezone.utc).isoformat()

    generated_ids = output_ids[0][prompt_tokens:]
    completion_tokens = int(generated_ids.shape[-1])
    output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    usage = {
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
    }
    return output, usage, latency_s, started_at, ended_at


def build_llama4_style_record(sample, spec, active_model_id, output, error, temperature, max_tokens, usage, latency_s, started_at, ended_at):
    completion_tokens = usage.get("completion_tokens") if usage else None
    tokens_per_second = round(completion_tokens / latency_s, 3) if completion_tokens and latency_s else None
    output_text = output or ""

    return {
        "sample_id": sample.get("sample_id"),
        "benchmark_scope": sample.get("benchmark_scope"),
        "category": sample.get("category"),
        "difficulty": sample.get("difficulty"),
        "topic": sample.get("topic"),
        "model_name": spec["model_name"],
        "model_id": active_model_id,
        "system_prompt": SYSTEM_PROMPT,
        "prompt": sample.get("prompt"),
        "model_output": output,
        "error": error,
        "status": "success" if error is None else "failed",
        "generation_params": {
            "temperature": temperature,
            "max_tokens": max_tokens,
        },
        "usage": usage,
        "metrics": {
            "latency_s": latency_s,
            "tokens_per_second": tokens_per_second,
            "output_char_count": len(output_text),
            "output_word_count": len(output_text.split()) if output_text else 0,
        },
        "metadata": {
            "started_at": started_at,
            "ended_at": ended_at,
        },
    }

In [ ]:
def run_one_model(spec: dict, samples: list[dict]) -> list[dict]:
    import torch

    active_model_id = model_ref(spec)
    print(f"\nLoading {spec['model_name']}: {active_model_id}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        tokenizer, model = load_phi3_model(
            model_id_or_path=active_model_id,
            cache_dir=cache_dir,
            local_files_only=LOCAL_FILES_ONLY,
            device=DEVICE,
            torch_dtype_name=TORCH_DTYPE,
            load_in_4bit=LOAD_IN_4BIT,
            trust_remote_code=spec.get("trust_remote_code", False),
        )
    except Exception as exc:
        print("Model load failed:", exc)
        rows = []
        now = datetime.now(timezone.utc).isoformat()
        for sample in samples:
            temperature, max_tokens = parse_generation_params(sample.get("recommended_generation_params"))
            rows.append(
                build_llama4_style_record(
                    sample=sample,
                    spec=spec,
                    active_model_id=active_model_id,
                    output=None,
                    error=f"Model load failed: {exc}",
                    temperature=temperature,
                    max_tokens=max_tokens,
                    usage=None,
                    latency_s=0.0,
                    started_at=now,
                    ended_at=now,
                )
            )
        return rows

    rows = []
    try:
        for sample in tqdm(samples, desc=spec["model_name"]):
            temperature, max_tokens = parse_generation_params(sample.get("recommended_generation_params"))
            try:
                output, usage, latency_s, started_at, ended_at = generate_phi3_response(
                    tokenizer=tokenizer,
                    model=model,
                    prompt=sample["prompt"],
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                error = None
            except Exception as exc:
                output = None
                usage = None
                error = str(exc)
                latency_s = 0.0
                started_at = datetime.now(timezone.utc).isoformat()
                ended_at = started_at

            rows.append(
                build_llama4_style_record(
                    sample=sample,
                    spec=spec,
                    active_model_id=active_model_id,
                    output=output,
                    error=error,
                    temperature=temperature,
                    max_tokens=max_tokens,
                    usage=usage,
                    latency_s=latency_s,
                    started_at=started_at,
                    ended_at=ended_at,
                )
            )

            if SLEEP_SECONDS:
                time.sleep(SLEEP_SECONDS)
    finally:
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return rows


all_results = []
for spec in MODEL_SPECS:
    model_rows = run_one_model(spec, samples)
    all_results.extend(model_rows)
    save_json(model_rows, OUTPUT_DIR / f"{safe_filename(spec['model_name'])}_outputs.json")
    save_json(all_results, OUTPUT_DIR / "phi3_benchmark_outputs.json")

if COPY_OUTPUTS_TO_DRIVE and IN_COLAB:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.glob("*.json"):
        shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)
    print("Copied outputs to Drive:", DRIVE_OUTPUT_DIR)

print("Done. Total rows:", len(all_results))

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*.json")):
    rows = json.loads(path.read_text(encoding="utf-8"))
    print(path.name, "rows=", len(rows))
    if rows:
        first = rows[0]
        print("  model:", first.get("model_name"), "status:", first.get("status"))
        print("  keys:", list(first.keys()))